In [1]:
import pandas as pd
import sqlite3

# Load cleaned dataset
df = pd.read_csv('../data/superstore_cleaned.csv')

# Connect to (or create) SQLite database
conn = sqlite3.connect('../data/superstore.db')

# Write DataFrame to SQL table 'sales'
df.to_sql('sales', conn, if_exists='replace', index=False)

print("Database 'superstore.db' initialized successfully with 'sales' table!")

Database 'superstore.db' initialized successfully with 'sales' table!


In [2]:
# Query 1: Top 10 Most Profitable Products
query_top = """
SELECT 
    product_name,
    category,
    sub_category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM sales
GROUP BY product_name
ORDER BY total_profit DESC
LIMIT 10;
"""

top_products = pd.read_sql_query(query_top, conn)
print("--- TOP 10 MOST PROFITABLE PRODUCTS ---")
display(top_products)

# Query 2: Top 10 Biggest Loss-Making Products
query_bottom = """
SELECT 
    product_name,
    category,
    sub_category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM sales
GROUP BY product_name
ORDER BY total_profit ASC
LIMIT 10;
"""

bottom_products = pd.read_sql_query(query_bottom, conn)
print("\n--- TOP 10 BIGGEST LOSS-MAKING PRODUCTS ---")
display(bottom_products)

--- TOP 10 MOST PROFITABLE PRODUCTS ---


,product_name,category,sub_category,total_sales,total_profit
0,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,61599.82,25199.93
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,27453.38,7753.04
2,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,18839.69,6983.88
3,Canon PC1060 Personal Laser Copier,Technology,Copiers,11619.83,4570.93
4,HP Designjet T520 Inkjet Large Format Printer ...,Technology,Machines,18374.90,4094.98
5,Ativa V4110MDD Micro-Cut Shredder,Technology,Machines,7699.89,3772.95
6,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,14299.89,3717.97
7,Plantronics Savi W720 Multi-Device Wireless He...,Technology,Accessories,9367.29,3696.28
8,Ibico EPK-21 Electric Binding System,Office Supplies,Binders,15875.92,3345.28
9,Zebra ZM400 Thermal Label Printer,Technology,Machines,6965.70,3343.54



--- TOP 10 BIGGEST LOSS-MAKING PRODUCTS ---


,product_name,category,sub_category,total_sales,total_profit
0,Cubify CubeX 3D Printer Double Head Print,Technology,Machines,11099.96,-8879.97
1,Lexmark MX611dhe Monochrome Laser Printer,Technology,Machines,16829.90,-4589.97
2,Cubify CubeX 3D Printer Triple Head Print,Technology,Machines,7999.98,-3839.99
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,Tables,9917.64,-2876.12
4,Bush Advantage Collection Racetrack Conference...,Furniture,Tables,9544.73,-1934.40
5,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,17965.07,-1878.17
6,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,22638.48,-1811.08
7,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,Supplies,16656.20,-1299.18
8,Balt Solid Wood Round Tables,Furniture,Tables,6518.75,-1201.06
9,BoxOffice By Design Rectangular and Half-Moon ...,Furniture,Tables,1706.25,-1148.44


In [3]:
# Query 3: Regional Sales, Profitability, and Average Discount
query_region = """
SELECT 
    region,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(discount) * 100, 2) AS avg_discount_pct
FROM sales
GROUP BY region
ORDER BY total_profit DESC;
"""

region_performance = pd.read_sql_query(query_region, conn)
print("--- REGIONAL PERFORMANCE ANALYSIS ---")
display(region_performance)

--- REGIONAL PERFORMANCE ANALYSIS ---


,region,total_orders,total_sales,total_profit,avg_discount_pct
0,West,1611,725457.82,108418.45,10.93
1,East,1401,678781.24,91522.78,14.54
2,South,822,391721.91,46749.43,14.73
3,Central,1175,501239.89,39706.36,24.04


In [4]:
# Query 4: Top 5 Customers Per Region using Window Functions
query_customers = """
WITH Customer_Sales AS (
    SELECT 
        region,
        customer_id,
        customer_name,
        ROUND(SUM(sales), 2) AS total_spent,
        ROUND(SUM(profit), 2) AS total_profit_generated,
        DENSE_RANK() OVER (PARTITION BY region ORDER BY SUM(sales) DESC) AS rank
    FROM sales
    GROUP BY region, customer_id, customer_name
)
SELECT 
    region,
    rank,
    customer_id,
    customer_name,
    total_spent,
    total_profit_generated
FROM Customer_Sales
WHERE rank <= 5
ORDER BY region, rank;
"""

top_customers_by_region = pd.read_sql_query(query_customers, conn)
print("--- TOP 5 CUSTOMERS BY REGION ---")
display(top_customers_by_region)

--- TOP 5 CUSTOMERS BY REGION ---


,region,rank,customer_id,customer_name,total_spent,total_profit_generated
0,Central,1,TC-20980,Tamara Chand,18437.14,8745.06
1,Central,2,AB-10105,Adrian Barton,12181.59,5362.61
2,Central,3,BM-11140,Becky Martin,10539.90,-1878.79
3,Central,4,SC-20095,Sanjit Chand,9900.19,4668.69
4,Central,5,HM-14860,Harry Marie,6621.48,1996.88
5,East,1,TA-21385,Tom Ashbrook,13723.50,4599.21
6,East,2,HL-15040,Hunter Lopez,10522.55,5045.86
7,East,3,BS-11365,Bill Shonely,10022.29,2558.58
8,East,4,GT-14710,Greg Tran,9382.93,1749.11
9,East,5,SV-20365,Seth Vernon,9216.57,1356.88


In [5]:
# Close SQLite connection
conn.close()
print("Database connection closed successfully.")

Database connection closed successfully.
